In [1]:
from functools import partial
import contextlib
from pathlib import Path

import jax
import jax.numpy as jnp
import jax.experimental.pallas as pl
import jax.experimental.pallas.tpu as pltpu

In [2]:
@contextlib.contextmanager
def profile(path="/tmp/profiles", port=8791):
  with jax.profiler.trace("/tmp/profiles"):
    yield
  profiles = sorted(Path(path).absolute().glob("**/*.xplane.pb"), key=lambda x: x.stat().st_mtime)
  profile_name = profiles[-1].parts[-2]
  if port == 8791:
    url = "http://localhost:{port}/data/plugin/profile/trace_viewer@;run={name};tag=trace_viewer@"  # xprof version
  else:
    url = "http://localhost:{port}/?run={name}&tag=trace_viewer"  # tensorboard version
  print(url.format(port=port, name=profile_name))

In [4]:
x = jnp.zeros((128, 8, 1024))
y = jnp.arange(128)[:, None, None] * jnp.ones((128, 8, 512))
offset = jax.random.randint(jax.random.key(0), y.shape[0], minval=0, maxval=(x.shape[-1] - y.shape[-1]) // 128 + 1)

In [5]:
@jax.jit
def scatter(x, y, offset):
  idx = (offset[:, None, None] * 128 + jnp.arange(y.shape[-1])[None, None, :]) * jnp.ones(y.shape[1], "int32")[None, :, None]
  return jnp.put_along_axis(x, idx, y, axis=-1, inplace=False)

In [6]:
@jax.jit
def scatter_ds(x, y, offset):
  return jax.vmap(partial(jax.lax.dynamic_update_slice_in_dim, axis=-1))(x, y, 128 * offset)

In [11]:
@partial(jax.jit, donate_argnames=("x",))
def scatter_pallas(x, y, offset):
  x_ref, y_ref, offset_ref = jax.tree.map(jax.new_ref, (x, y, offset))
  
  @pl.core_map(mesh=pltpu.create_tensorcore_mesh("core"))
  def _():
    @partial(pl.run_scoped, offset_smem=pltpu.SMEM(offset_ref.shape, offset_ref.dtype), sems=pltpu.SemaphoreType.DMA(offset_ref.shape))
    def _(offset_smem, sems):
      pltpu.sync_copy(offset_ref, offset_smem)
      copies = []
      for i in range(x.shape[0]):
        copy = pltpu.async_copy(y_ref.at[i, ...], x_ref.at[i, :, pl.ds(offset_smem[i] * 128, y.shape[-1])], sems.at[i])
        copies.append(copy)
      [copy.wait() for copy in copies]

  return x_ref[...]

In [8]:
from jax.experimental.compute_on import compute_on

@jax.jit
@compute_on("tpu_sparsecore")
def scatter_sc(x, y, offset):
  x_shape = x.shape
  idx = (offset[:, None] * 128 + jnp.arange(y.shape[-2] * y.shape[-1])[None, :])
  x = x.reshape((x.shape[0], -1))
  y = y.reshape((y.shape[0], -1))
  return x.at[:, idx].set(y).reshape(x_shape)

In [10]:
x = jax.block_until_ready(scatter(x, y, offset))
#x = jax.block_until_ready(scatter_sc(x, y, offset))
x = jax.block_until_ready(scatter_ds(x, y, offset))
x = jax.block_until_ready(scatter_pallas(x, y, offset))

with profile():
  x = jax.block_until_ready(scatter(x, y, offset))
  #x = jax.block_until_ready(scatter_sc(x, y, offset))
  x = jax.block_until_ready(scatter_ds(x, y, offset))
  x = jax.block_until_ready(scatter_pallas(x, y, offset))

http://localhost:8791/data/plugin/profile/trace_viewer@;run=2025_11_21_00_28_46;tag=trace_viewer@


In [12]:
jax.block_until_ready(scatter_pallas(x, y, offset))

Array([[[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        ...,
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       [[  1.,   1.,   1., ...,   0.,   0.,   0.],
        [  1.,   1.,   1., ...,   0.,   0.,   0.],
        [  1.,   1.,   1., ...,   0.,   0.,   0.],
        ...,
        [  1.,   1.,   1., ...,   0.,   0.,   0.],
        [  1.,   1.,   1., ...,   0.,   0.,   0.],
        [  1.,   1.,   1., ...,   0.,   0.,   0.]],

       [[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        ...,
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       ...,

       [[125., 125., 125

In [13]:
x

RuntimeError: Array has been deleted with shape=float32[128,8,1024].